In [46]:
!pip install bayesian-torch


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from bayesian_torch.models.dnn_to_bnn import get_kl_loss
from bayesian_torch.layers.variational_layers import LinearReparameterization
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)

Library Versions:
numpy: 2.5.1
pandas: 3.0.5
torch: 2.13.0+cpu


In [48]:

n_epochs = 100
verbose_option = True

# Regression for Naval Plant Maintenance

Load dataset

In [49]:
npm = pd.read_csv('navalplantmaintenance.csv',header=None)
npm_train, npm_test = train_test_split(npm,test_size=0.25,random_state=42)
npm_train_np = npm_train.to_numpy()
npm_test_np = npm_test.to_numpy()
x_train_np = npm_train_np[:,:16]
x_test_np = npm_test_np[:,:16]
y_train_np = npm_train_np[:,17]
y_test_np = npm_test_np[:,17]
x_mu = x_train_np.mean(axis=0)
x_sigma = x_train_np.std(axis=0)
y_mu=y_train_np.mean(axis=0)
y_sigma=y_train_np.std(axis=0)
def scale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return (x-x_mu)/x_sigma
def unscale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return x_sigma*x+x_mu
x_train_np_z = scale(x_train_np,x_mu,x_sigma)
y_train_np_z = scale(y_train_np,y_mu,y_sigma)
x_test_np_z = scale(x_test_np,x_mu,x_sigma)
y_test_np_z = scale(y_test_np,y_mu,y_sigma)
x_train_t_z = torch.FloatTensor(x_train_np_z)
y_train_t_z = torch.FloatTensor(y_train_np_z)
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)

1. Using PyTorch, perform variational inference using a mean-field Gaussian variational distribution and reparamaterization layers for Gaussian heteroscedastic regression.

In [50]:
def nlls(y, mu, std):
    return torch.square(y - mu)/(2.0*torch.square(std))+torch.log(std)

class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = LinearReparameterization(inputSize, hiddenSize) # Call the PyTorch Linear Local Reparameterization layer constructor going from inputSize to hiddenSize
        self.layer2 = LinearReparameterization(hiddenSize, hiddenSize) # Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to hiddenSize
        self.linear_mu = LinearReparameterization(hiddenSize, outputSize) # Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to outputSize
        self.linear_sigma = LinearReparameterization(hiddenSize, outputSize) # Call the PyTorch Linear Local Reparameterization layer constructor going from hiddenSize to outputSize

    def forward(self, x):
        h1 = torch.nn.functional.relu(self.layer1(x,return_kl=False))
        h2 = torch.nn.functional.relu(self.layer2(h1,return_kl=False))
        mu = self.linear_mu(h2,return_kl=False)
        sigma = torch.nn.functional.softplus(self.linear_sigma(h2,return_kl=False))
        return mu, sigma

In [51]:
x_train_t_z

tensor([[-1.5340e+00, -1.5500e+00, -1.1291e+00,  ..., -9.4888e-01,
         -1.5078e-01, -9.5088e-01],
        [-1.5340e+00, -1.5500e+00, -1.0912e+00,  ..., -9.4888e-01,
         -1.3044e+00, -1.0119e+00],
        [ 7.5376e-01,  7.7377e-01,  5.2904e-01,  ...,  9.4549e-01,
          3.8730e-01,  4.0011e-01],
        ...,
        [ 1.5722e+00,  1.5484e+00,  2.0532e+00,  ...,  1.8927e+00,
          2.2106e+00,  2.2395e+00],
        [ 3.8452e-01,  3.8648e-01,  1.1418e-01,  ..., -1.6933e-03,
          3.1161e-02,  3.7743e-02],
        [ 1.1573e+00,  1.1611e+00,  1.0702e+00,  ...,  9.4549e-01,
          1.0609e+00,  1.0776e+00]])

In [52]:
model = nn(x_train_np_z.shape[1],25, 1)

optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-3)

for i in range(n_epochs):
    mu, s = model(x_train_t_z)
    M = mu.shape[0]
    nll_loss = nlls(y_train_t_z, mu, s).mean()
    kl = get_kl_loss(model)
    loss = nll_loss + 1/x_train_np.shape[1] * kl # The VI loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()  
    if verbose_option: print(i, loss)

0 tensor(1.9146, grad_fn=<AddBackward0>)
1 tensor(1.9126, grad_fn=<AddBackward0>)
2 tensor(1.9403, grad_fn=<AddBackward0>)
3 tensor(1.9219, grad_fn=<AddBackward0>)
4 tensor(1.9024, grad_fn=<AddBackward0>)
5 tensor(1.9965, grad_fn=<AddBackward0>)
6 tensor(1.8729, grad_fn=<AddBackward0>)
7 tensor(1.8725, grad_fn=<AddBackward0>)
8 tensor(1.8942, grad_fn=<AddBackward0>)
9 tensor(1.8959, grad_fn=<AddBackward0>)
10 tensor(1.8634, grad_fn=<AddBackward0>)
11 tensor(1.8729, grad_fn=<AddBackward0>)
12 tensor(1.8854, grad_fn=<AddBackward0>)
13 tensor(1.9040, grad_fn=<AddBackward0>)
14 tensor(1.7961, grad_fn=<AddBackward0>)
15 tensor(1.8295, grad_fn=<AddBackward0>)
16 tensor(1.8191, grad_fn=<AddBackward0>)
17 tensor(1.8495, grad_fn=<AddBackward0>)
18 tensor(1.8320, grad_fn=<AddBackward0>)
19 tensor(1.8748, grad_fn=<AddBackward0>)
20 tensor(1.8645, grad_fn=<AddBackward0>)
21 tensor(1.8967, grad_fn=<AddBackward0>)
22 tensor(1.8727, grad_fn=<AddBackward0>)
23 tensor(1.8900, grad_fn=<AddBackward0>)
24

2.  Compute the mean and standard deviation predictions for 20 MC sampled models

In [53]:
mc_samples = 20
n_test_examples = y_test_np.shape[0]
y_test_mus_z = np.zeros([mc_samples,n_test_examples,1])
y_test_sigmas_z = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    mu, s = model(x_test_t_z) # Get prediction for one sampled parameter vector
    y_test_mus_z[i] = mu.detach().numpy() # Get the predicted probabilities of the test examples given the sampled parameter vector - this comment doesn't make sense when looking at the code, so I am doing what the code seems to ask for.
    y_test_sigmas_z[i] = s.detach().numpy()  # Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector. This comment doesn't make sense when looking at the code, so I am doing what the code seems to ask for.

In [54]:
y_test_mus_z

array([[[ 0.11689536],
        [ 0.12283147],
        [ 0.1143803 ],
        ...,
        [ 0.12183885],
        [ 0.13729489],
        [ 0.12291765]],

       [[ 0.14350536],
        [ 0.12290134],
        [ 0.11274839],
        ...,
        [ 0.12146134],
        [ 0.14156109],
        [ 0.12309786]],

       [[ 0.10266578],
        [ 0.04752687],
        [ 0.02444258],
        ...,
        [ 0.04573871],
        [ 0.12829466],
        [ 0.04777154]],

       ...,

       [[ 0.05990184],
        [ 0.15987721],
        [ 0.14180857],
        ...,
        [ 0.16298753],
        [ 0.0353617 ],
        [ 0.15957391]],

       [[ 0.05410804],
        [-0.05309243],
        [-0.01210503],
        ...,
        [-0.05155668],
        [ 0.05981949],
        [-0.05332527]],

       [[-0.00584243],
        [-0.04823691],
        [-0.0501271 ],
        ...,
        [-0.0529508 ],
        [ 0.0039316 ],
        [-0.04802038]]], shape=(20, 2984, 1))

3. Compute the Mean Squared Error (MSE) for the Gaussian VI model with Local Reparameterzaton layers for the test data using 20 MC samples and Bayesian model averaging.

In [55]:
y_test_mu_z = y_test_mus_z.mean(axis=0) # Compute the mean predictions using Bayesian model averaging
y_test_mu = unscale(y_test_mu_z,y_mu,y_sigma)
print('MSE:', mean_squared_error(y_test_np, y_test_mu))

MSE: 5.7058542616593566e-05


In [56]:
y_test_mu

array([[0.98804639],
       [0.98798843],
       [0.98786107],
       ...,
       [0.98798003],
       [0.98808596],
       [0.98798883]], shape=(2984, 1))

4. Compute the aleatoric uncertainty for each regression test example for VI model.

In [57]:
y_test_mus = unscale(y_test_mus_z,y_mu,y_sigma)
y_test_sigmas = y_test_sigmas_z * y_sigma
y_test_aleatoric = np.mean(np.square(y_test_sigmas), axis=0) # Compute the aleatoric uncertainties
mean_prediction = np.mean(y_test_mus, axis=0)
y_test_epistemic = np.mean(np.square(y_test_mus - mean_prediction), axis=0) # Compute the epistemic uncertainties

In [58]:
y_test_aleatoric

array([[5.84086481e-05],
       [6.80484204e-05],
       [4.73120809e-05],
       ...,
       [6.81681844e-05],
       [6.53716186e-05],
       [6.80636721e-05]], shape=(2984, 1))

In [59]:
y_test_epistemic

array([[2.58840128e-07],
       [5.39363900e-07],
       [3.33706195e-07],
       ...,
       [5.43474677e-07],
       [2.89608533e-07],
       [5.39548106e-07]], shape=(2984, 1))

# Classification for Ship Detection


Load Ship Detection Dataset

In [60]:
import torch 
from torch.utils.data import Dataset, DataLoader
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from bayesian_torch.layers.flipout_layers import LinearFlipout 
from torchvision.io import read_image
from torch.utils.data import random_split
from torchvision.transforms.functional import resize
from sklearn import preprocessing
import numpy as np
from pathlib import Path
import torchmetrics

ROOT_PATH = "shipsnet"
LR = 1e-4
IMG_SIZE = [80]

tensor_size = IMG_SIZE[0]**2 * 3

def max_scaling(image):
    image = image / 255.0
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    return image

def normalize_img(image):
    means = torch.Tensor([[[105.0385]],[[108.1886]],[[ 94.9558]]])
    stds = torch.Tensor([[[48.4294]],[[40.0104]],[[38.6445]]])
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    image = image - means
    image = image / stds
    return image

#https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
class ShipDataset(Dataset):
    def __init__(self, root_path, transform = None):
        self.root_path = Path(root_path)
        self.files = list(self.root_path.rglob("*/*"))
        self.classes = list(set([int(entry.parts[-1]) for entry in self.root_path.rglob("*") if Path(entry).is_dir()]))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = read_image(str(self.files[idx]))
        label = int(self.files[idx].parts[-2])
        if self.transform:
            image = self.transform(image)
        return image, label
    

full_dataset  = ShipDataset(ROOT_PATH, transform = normalize_img,)
n_classes = len(full_dataset.classes)
print("Length of full dataset: ", len(full_dataset), " - with " ,n_classes ," classes. ")
train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2])
print("Length of training dataset: ", len(train_dataset))
print("Length of Test dataset: ", len(test_dataset))

train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset))

criterion = torch.nn.BCELoss(reduce='mean')
accuracy = torchmetrics.classification.BinaryAccuracy()


Length of full dataset:  4000  - with  2  classes. 
Length of training dataset:  3200
Length of Test dataset:  800


C:\Users\shai1\PyCharmMiscProject\.venv\Lib\site-packages\torch\nn\modules\loss.py:48: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


5. Using PyTorch, perform variational inference using a mean-field Gaussian variational distribution and flipout layers for non-linear binary classification

In [61]:
class logistic(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(logistic, self).__init__()
        self.layer1 = LinearFlipout(inputSize, hiddenSize) # Call the PyTorch Linear Flipout layer constructor going from inputSize to hiddenSize
        self.layer2 = LinearFlipout(hiddenSize, hiddenSize) # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.layer3 = LinearFlipout(hiddenSize, hiddenSize) # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.layer4 = LinearFlipout(hiddenSize, hiddenSize) # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to hiddenSize
        self.p = LinearFlipout(hiddenSize, outputSize) # Call the PyTorch Linear Flipout layer constructor going from hiddenSize to outputSize

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = torch.nn.functional.relu(self.layer1(x,return_kl=False))
        x = torch.nn.functional.relu(self.layer2(x,return_kl=False))
        x = torch.nn.functional.relu(self.layer3(x,return_kl=False))
        x = torch.nn.functional.relu(self.layer4(x,return_kl=False))
        x = self.p(x,return_kl=False)
        return torch.sigmoid(x)

In [62]:
model = logistic(tensor_size, 50, 1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(n_epochs):
    for data, label in train_dataloader:
        p = model(data)
        M = p.shape[0]
        nll_loss = criterion(p.squeeze(),label*1.0) 
        kl = get_kl_loss(model)
        loss = nll_loss + 1/data.shape[1] * kl # Write the VI loss
        acc = accuracy(p.squeeze(), label)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if verbose_option: print(epoch, loss, acc, end="/r")
    print(epoch)

0 tensor(9.3823, grad_fn=<AddBackward0>) tensor(0.3862)/r0
1 tensor(9.3337, grad_fn=<AddBackward0>) tensor(0.4206)/r1
2 tensor(9.3000, grad_fn=<AddBackward0>) tensor(0.4403)/r2
3 tensor(9.2571, grad_fn=<AddBackward0>) tensor(0.4491)/r3
4 tensor(9.2301, grad_fn=<AddBackward0>) tensor(0.4647)/r4
5 tensor(9.1952, grad_fn=<AddBackward0>) tensor(0.5053)/r5
6 tensor(9.1775, grad_fn=<AddBackward0>) tensor(0.5191)/r6
7 tensor(9.1540, grad_fn=<AddBackward0>) tensor(0.5425)/r7
8 tensor(9.1299, grad_fn=<AddBackward0>) tensor(0.5725)/r8
9 tensor(9.1381, grad_fn=<AddBackward0>) tensor(0.5666)/r9
10 tensor(9.1133, grad_fn=<AddBackward0>) tensor(0.5931)/r10
11 tensor(9.0832, grad_fn=<AddBackward0>) tensor(0.6137)/r11
12 tensor(9.0757, grad_fn=<AddBackward0>) tensor(0.6366)/r12
13 tensor(9.0720, grad_fn=<AddBackward0>) tensor(0.6378)/r13
14 tensor(9.0586, grad_fn=<AddBackward0>) tensor(0.6438)/r14
15 tensor(9.0530, grad_fn=<AddBackward0>) tensor(0.6522)/r15
16 tensor(9.0384, grad_fn=<AddBackward0>) te

6.  Compute the predicted probabilities and entropy predictions for 20 MC sampled models

In [71]:
mc_samples = 20
n_test_examples = len(test_dataset)
y_test_probs = np.zeros([mc_samples,n_test_examples,1])
y_test_entropies = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    for data, label in test_dataloader:
        results = model(data).detach().numpy()
        y_test_probs[i] = results # Get the predicted means for one sampled parameter vector
        y_test_entropies[i] += np.array([-(x*np.log(x) + (1-x) * np.log(1-x)) for x in results]) # Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector

7. Compute the Bayesian model averaging predictions for each classification test example for the VI model.

In [72]:
y_test_probs_avg = np.mean(y_test_probs, axis=0) # Compute Bayesian model averaging predictions

In [73]:
y_test_probs_avg

array([[0.81084092],
       [0.07157844],
       [0.61193292],
       [0.07894304],
       [0.45922033],
       [0.38207703],
       [0.34623796],
       [0.20048389],
       [0.08134498],
       [0.32341292],
       [0.22720728],
       [0.01967375],
       [0.71788383],
       [0.02295639],
       [0.42920554],
       [0.17954477],
       [0.48195097],
       [0.02959707],
       [0.32158307],
       [0.44851753],
       [0.22119171],
       [0.61945455],
       [0.11551365],
       [0.67385627],
       [0.13344755],
       [0.41233713],
       [0.19787877],
       [0.22525999],
       [0.23306617],
       [0.12077843],
       [0.25855561],
       [0.4693928 ],
       [0.28164446],
       [0.20905966],
       [0.83394201],
       [0.13076288],
       [0.0210501 ],
       [0.32308977],
       [0.02197467],
       [0.10719062],
       [0.0314369 ],
       [0.74622533],
       [0.38052883],
       [0.77183093],
       [0.76578957],
       [0.06133539],
       [0.68904887],
       [0.286

8. Compute the aleatoric uncertainty for each classification test example for the Gaussian VI model.

In [74]:
y_test_aleatoric  = np.mean(y_test_entropies, axis=0) # Computer aleatoric uncertainty

In [75]:
y_test_aleatoric

array([[0.40396052],
       [0.23140355],
       [0.57403521],
       [0.2088202 ],
       [0.59890104],
       [0.58672431],
       [0.58606993],
       [0.47594114],
       [0.21991531],
       [0.57472138],
       [0.52066502],
       [0.08563028],
       [0.56208837],
       [0.09160204],
       [0.65510198],
       [0.41248455],
       [0.57443687],
       [0.10718535],
       [0.58685025],
       [0.66841333],
       [0.44502211],
       [0.58970699],
       [0.32082741],
       [0.58760366],
       [0.3295278 ],
       [0.63965645],
       [0.47459162],
       [0.44821225],
       [0.4923238 ],
       [0.33254486],
       [0.54971756],
       [0.64543995],
       [0.54389667],
       [0.48635141],
       [0.37084261],
       [0.27266214],
       [0.08981727],
       [0.62040591],
       [0.07313812],
       [0.31188124],
       [0.10690153],
       [0.51032846],
       [0.64283822],
       [0.5128238 ],
       [0.50189935],
       [0.20509524],
       [0.59498578],
       [0.552

9. Compute the epistemic uncertainty for each classification test example for the Gaussian VI model.

In [76]:
y_test_uncertainty = np.array([-(x*np.log(x) + (1-x) * np.log(1-x)) for x in y_test_probs_avg]) # Compute the total uncertainty
y_test_epistemic =  y_test_uncertainty - y_test_aleatoric # Compute the epistemic uncertainty

In [77]:
y_test_epistemic

array([[0.08104079],
       [0.02629931],
       [0.0938404 ],
       [0.06736007],
       [0.09091648],
       [0.07834747],
       [0.05901671],
       [0.02513137],
       [0.06212674],
       [0.05469393],
       [0.01521477],
       [0.01113638],
       [0.0328525 ],
       [0.01772987],
       [0.02798773],
       [0.05821794],
       [0.11805863],
       [0.0261534 ],
       [0.04120673],
       [0.01942356],
       [0.08339002],
       [0.07462353],
       [0.03706249],
       [0.04381032],
       [0.06336041],
       [0.03804144],
       [0.02285605],
       [0.08527294],
       [0.05063096],
       [0.03592824],
       [0.02182314],
       [0.04583246],
       [0.05060308],
       [0.02635671],
       [0.07873764],
       [0.11517221],
       [0.01228098],
       [0.00877063],
       [0.03248965],
       [0.02871979],
       [0.03280052],
       [0.05611572],
       [0.0214842 ],
       [0.02423073],
       [0.04241466],
       [0.02553095],
       [0.02487379],
       [0.046

In [78]:
from sklearn.metrics import classification_report

for data, label in test_dataloader:
    print(classification_report(label.flatten(), y_test_probs_avg.round().flatten()))

              precision    recall  f1-score   support

           0       0.90      0.96      0.93       602
           1       0.83      0.67      0.74       198

    accuracy                           0.89       800
   macro avg       0.86      0.81      0.83       800
weighted avg       0.88      0.89      0.88       800

